# 14.3 임베딩을 실전에 쓰기: Transformer 임베딩과 RAG — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter14_3_embeddings_rag.ipynb)

책 본문: [14.3 임베딩을 실전에 쓰기](https://smhanlab.com/book-ml/kor/ml1/chapter14.html)

작은 문서 집합에서 임베딩 코사인 유사도로 검색하는 미니 RAG 데모를 직접 만들어봅니다.


이 노트북은 14.3절의 **밀집 검색 → RAG의 검색(retrieval)단계**을 코드로 끝까지 실행합니다.

- (1) 14.2의 `train_skipgram` + `cos_sim`을 그대로 쓰고, 문서 벡터를 단어 벡터의 **평균**(`mean_vec`)으로 만든다.
- (2) 5개 문서를 임베딩하고, 두 개의 질문으로 코사인 유사도 상위 문서를 검색한다(본문의 숫자와 일치해야 한다).
- (3) 문서 임베딩을 2D로 투영해 "의미가 비슷한 문서가 가까이"를 눈으로 본다.
- (4) 질문별 유사도 막대그래프로 1위가 선명하게 분리되는지 확인한다.
- (5) "내적 vs 코사인" — 벡터를 2배로 키워도 코사인은 변하지 않음을 확인한다(본문 "자주 하는 실수").

> **Caveat:** `train_skipgram`은 14.2의 재현 코드를 그대로 쓴다(실전 RAG는 전문 문서 임베딩 모델을 쓰지만, 파이프라인 구조는 이 20줄과 같다).

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

IMG = "/home/smhan/book-ml/kor/src/images"   # (Colab에서는 /tmp로 바꾸면 됨)
os.makedirs(IMG, exist_ok=True)


## 1. 임베딩 도구: `train_skipgram` + `cos_sim` + `mean_vec`

본문 `mean_vec`이 **문서 벡터 = 단어 벡터의 평균**을 만든다. `train_skipgram`과 `cos_sim`은 14.2 노트북과 동일하다.

In [ ]:
import math, random

def train_skipgram(corpus, window=2, dim=8, epochs=50, lr=0.05, neg_k=3):
    # 14.2와 동일한 재현 코드: skip-gram + negative sampling
    vocab = sorted(set(corpus))
    idx = {w: i for i, w in enumerate(vocab)}
    V = len(vocab)
    W_in  = [[random.uniform(-0.5, 0.5) for _ in range(dim)] for _ in range(V)]
    W_out = [[random.uniform(-0.5, 0.5) for _ in range(dim)] for _ in range(V)]

    def sigmoid(z):
        return 1 / (1 + math.exp(-max(-20, min(20, z))))

    pairs = []
    for i, center in enumerate(corpus):
        for j in range(max(0, i - window), min(len(corpus), i + window + 1)):
            if j != i:
                pairs.append((idx[center], idx[corpus[j]]))

    for _ in range(epochs):
        random.shuffle(pairs)
        for c, o in pairs:
            for t, label in [(o, 1)] + [(random.randrange(V), 0) for _ in range(neg_k)]:
                z = sum(W_in[c][k] * W_out[t][k] for k in range(dim))
                grad = (sigmoid(z) - label) * lr
                for k in range(dim):
                    g_in, g_out = W_in[c][k], W_out[t][k]
                    W_in[c][k]  -= grad * g_out
                    W_out[t][k] -= grad * g_in
    return {w: W_in[idx[w]] for w in vocab}

def cos_sim(a, b):
    # 14.2와 동일
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(x * x for x in b))
    return dot / (na * nb + 1e-9)

def mean_vec(words, emb):
    # NEW: 문서 벡터 = (어휘에 있는) 단어 벡터의 평균
    dim = len(next(iter(emb.values())))
    present = [w for w in words if w in emb]
    if not present:
        return None
    v = [0.0] * dim
    for w in present:
        for k in range(dim):
            v[k] += emb[w][k]
    return [x / len(present) for x in v]

tok = lambda s: s.replace(". ", " ").split()
print("함수 준비 완료: train_skipgram, cos_sim, mean_vec")

## 2. 5개 문서 임베딩

에너지·동물·컴퓨팅 주제 5개 문서를 하나의 토큰 스트림으로 학습(14.2의 word2vec을 문서 토큰에 적용). `random.seed(42)`로 재현 가능.

In [ ]:
corpus_docs = {
    "D1 전기차": "전기차는 배터리로 움직인다. 전기차 충전은 연료 대신 전기를 쓴다. 배터리 용량이 클수록 전기차 주행거리가 늘어난다.",
    "D2 태양광": "태양광 발전은 햇빛으로 전기를 만든다. 태양광 패널은 에너지를 공급한다. 화석 연료 대신 태양광이 환경에 좋다.",
    "D3 양자": "양자 컴퓨터는 큐비트로 정보를 처리한다. 큐비트는 중첩 상태에 있을 수 있다. 양자 알고리즘은 슈퍼포지션을 활용한다.",
    "D4 고양이": "고양이는 애묘인이 좋아하는 동물이다. 고양이는 털을 손질한다. 강아지도 반려동물로 인기지만 고양이는 독립적이다.",
    "D5 배터리": "배터리와 태양광 발전은 에너지 저장과 생산을 함께 한다. 전기차 충전은 배터리 기술에 의존한다.",
}
docnames = list(corpus_docs)
flat = []
for n, t in corpus_docs.items():
    flat += tok(t)
print("토큰 수:", len(flat), " 고유어 수:", len(set(flat)))

random.seed(42)   # 재현 가능하게
emb = train_skipgram(flat, window=2, dim=8, epochs=400, lr=0.1, neg_k=3)
docvecs = {n: mean_vec(tok(t), emb) for n, t in corpus_docs.items()}
print("문서 임베딩(8차원) 생성 완료:", {n: round(len(docvecs[n])) for n in docnames})

## 3. 코사인 유사도 검색 (본문 실습)

두 질문으로 상위 5개를 검색한다. 본문의 출력(D3 양자 +0.793, D4 고양이 +0.951)과 일치해야 한다.

In [ ]:
def search(query, k=5):
    qv = mean_vec(tok(query), emb)
    return sorted(((n, cos_sim(qv, docvecs[n])) for n in docnames),
                  key=lambda kv: -kv[1])[:k]

for q in ["양자 컴퓨터", "고양이는 손질한다"]:
    print("\n질문:", q)
    for n, s in search(q):
        print(f"  {n}: {s:+.3f}")

# 확인: 본문과 일치해야 함
top1 = search("양자 컴퓨터")[0]
top2 = search("고양이는 손질한다")[0]
assert top1 == ("D3 양자", top1[1]) or top1[0] == "D3 양자", top1
assert top2[0] == "D4 고양이", top2
print("\n1위 검증: 양자 컴퓨터 ->", top1[0], "| 고양이는 손질한다 ->", top2[0])

## 4. 2D 투영: "의미가 비슷한 문서가 가까이"

문서 임베딩 8차원 → 2차원(SVD 상위 2 좌표). 에너지 문서(D1 전기차·D2 태양광·D5 배터리)와 비에너지(D3 양자·D4 고양이)가 분리되는지 본다.

In [ ]:
# 8차원 문서 임베딩을 2D로 투영(SVD 상위 2 좌표, 14.1의 PCA와 동일한 선형 투영)
M = np.array([docvecs[n] for n in docnames])
M2 = M - M.mean(axis=0)
u, s, vt = np.linalg.svd(M2, full_matrices=False)
P2 = M @ vt[:2].T

# data label을 영어로 (Colab CJK 폰트 안전망, 14.2 노트북과 같은 관례)
lbl = {"D1 전기차": "D1 EV", "D2 태양광": "D2 solar", "D3 양자": "D3 quantum",
       "D4 고양이": "D4 cat", "D5 배터리": "D5 battery"}
plt.figure(figsize=(5.5, 5))
for n, (x, y) in zip(docnames, P2):
    plt.scatter(x, y, s=150, zorder=3, color="#4c72b0")
    plt.annotate(lbl[n], (x, y), textcoords="offset points", xytext=(7, 5))
plt.title("5 doc embeddings (2D SVD projection)")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.tight_layout()
plt.savefig(IMG + "/ch14_3_doc_2d.svg")
plt.show()
print("에너지 문서(D1, D2, D5)와 비에너지 문서(D3, D4)가 분리되는지 확인")

## 5. 질문별 유사도: 1위가 선명하게 분리되는가

"양자 컴퓨터" 질문의 5개 문서 유사도를 막대로. 1위(D3)가 2위보다 훨씬 높으면, 그 1위가 검색의 정답이라는 신호.

In [ ]:
# 질문별 유사도 막대그래프: 1위가 2위보다 선명하게 높아야 한다
q = "양자 컴퓨터"
res = search(q)
vals = [s for _, s in res]
plt.figure(figsize=(6, 4))
colors = ["#4c72b0"] + ["#ddd5c3"] * (len(res) - 1)
plt.barh(range(len(res))[::-1], vals, color=colors)
plt.yticks(range(len(res))[::-1], [lbl[n] for n, _ in res])
plt.xlabel("cosine similarity")
plt.title("Query \"quantum computer\" vs 5 docs")
for i, (n, s) in enumerate(res):
    plt.text(s, len(res) - 1 - i, f" {s:+.3f}", va="center")
plt.tight_layout()
plt.savefig(IMG + "/ch14_3_rag_search.svg")
plt.show()
gap = res[0][1] - res[1][1]
print(f"1위-2위 간격: {gap:.3f} (0.3 이상이면 1위가 선명하게 분리)")
assert gap > 0.3, "간격이 너무 작음"

## 6. 내적 vs 코사인: "자주 하는 실수" 확인

문서 2 벡터를 2배로(\( (2,3) \to (4,6) \)) 키워도, 내적은 2배가 되고 코사인은 변하지 않는다.

In [ ]:
# 본문 "자주 하는 실수": 벡터를 2배로 키워도 코사인 유사도는 변하지 않는다
q2 = np.array([4.0, 1.0])     # 2D 손 계산 예제의 q
d2_a = np.array([2.0, 3.0])   # 문서 2 원래 벡터
d2_b = np.array([4.0, 6.0])   # = 2 * d2_a (같은 방향, 2배 크기)

print(f"내적  q . d2(원래) = {q2 @ d2_a:.1f}")
print(f"내적  q . d2(2배)  = {q2 @ d2_b:.1f}  <- 두 배로 커짐 (길이 편향)")
print(f"코사인 d2(원래)    = {cos_sim(q2, d2_a):.4f}")
print(f"코사인 d2(2배)     = {cos_sim(q2, d2_b):.4f}  <- 변하지 않음")
assert abs(cos_sim(q2, d2_a) - cos_sim(q2, d2_b)) < 1e-9
print("코사인 유사도는 크기에 불변(invariant) — 방향만 본다")

## 정리

- **밀집 검색** = 질문도 벡터, 문서도 벡터, 코사인 유사도로 상위 k개를 고르는 것 — 14.2의 단어 유사도 검색과 **동일한 연산**이 대상이 문서로 바뀐 것.
- 문서 벡터를 `mean_vec`(단어 벡터 평균)으로 만들면 단어 순서를 버린다(의도된 단순화, bag-of-words 한계).
- "양자 컴퓨터" 질문에서 1위 D3(0.793)과 2위(0.454) 사이에 0.34의 간격 — 1위가 선명하게 분리되면 검색이 "이 문구가 정답"이라는 신호를 줌.
- **내적은 각도 × 크기**라 문서가 길면 커지는 편향이 있고, **코사인은 각도만** 본다 — 그래서 유사도 검색에는 코사인(또는 단위벡터화 후 내적)을 쓴다.
- 이 데모는 RAG의 **검색**단계만이다. 검색된 문서를 받아 답을 **생성**하는단계는 LLM(Chapter 17)의 일.

**확인 문제 3**을 여기서 풀 수 있다: 질문을 "고양이는 애묘인이 좋아하는"으로 바꿔 1위가 여전히 D4인지(0.925), 그리고 1~2위 간격(0.271)이 "양자 컴퓨터" 때(0.339)보다 좁은지 확인하고, 간격이 좁아진다는 것이 검색 확신도에 어떤 의미인지 서술하라.
